In [21]:
import sys
import numpy as np
import tensorflow as tf
sys.path.append('..')
from src.LSTM import LSTM, binary_cross_entropy

import os
import json
import sys
from itertools import product
import numpy as np
import tensorflow as tf
sys.path.append('..')
from src.LSTM import LSTM
from src.metrics import evaluate_predictions

In [22]:
def pick_device():
    gpus = tf.config.list_physical_devices('GPU')
    if not gpus:
        print("No GPU visible to TensorFlow, using CPU")
        return '/CPU:0'
    try:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
        with tf.device('/GPU:0'):
            tf.reduce_sum(tf.random.uniform((8, 8))).numpy()
        print(f"Training on GPU: {gpus[0].name}")
        return '/GPU:0'
    except (RuntimeError, ValueError, tf.errors.OpError) as e:
        print(f"GPU unusable, falling back to CPU: {e}")
        return '/CPU:0'

DEVICE = pick_device()

No GPU visible to TensorFlow, using CPU


In [23]:
SEQ_LEN = 10
RBM_HIDDEN = 256
OUTPUT_DIM = 1001
BATCH = 8

In [24]:
net = LSTM(
    sequence_length=SEQ_LEN,
    input_dim=RBM_HIDDEN,
    output_dim=OUTPUT_DIM,
    hidden_units=128,
    dropout=0.2,
    learning_rate=1e-3
)
net.model.summary()

Model: "next_basket_lstm"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 10, 256)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 10, 256)   │          0 │ sequence_input[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pad_mask (Masking)  │ (None, 10, 256)   │          0 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_1 (Any)         │ (None, 10)        │          0 │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 128)       │    197,120 │ pad_mask[0][0],   │
│                     │                   │            │ any_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_norm           │ (None, 128)       │        256 │ lstm[0][0]        │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ seq_logits (Dense)  │ (None, 1001)      │    129,129 │ lstm_norm[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ basket_output       │ (None, 1001)      │          0 │ seq_logits[0][0]  │
│ (Activation)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 326,505 (1.25 MB)

 Trainable params: 326,505 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
assert net.model.input_shape == (None, SEQ_LEN, RBM_HIDDEN)
assert net.model.output_shape == (None, OUTPUT_DIM)
assert not any(isinstance(l, tf.keras.layers.Bidirectional) for l in net.model.layers)
print("Input and output shapes match the contract; LSTM is unidirectional")

Input and output shapes match the contract; LSTM is unidirectional


In [26]:
X_dummy = np.random.rand(BATCH, SEQ_LEN, RBM_HIDDEN).astype(np.float32)
y_dummy = (np.random.rand(BATCH, OUTPUT_DIM) < 0.01).astype(np.float32)

pred = net.model.predict(X_dummy, verbose=0)
assert pred.shape == (BATCH, OUTPUT_DIM)
assert pred.min() >= 0.0 and pred.max() <= 1.0
print(f"Prediction shape: {pred.shape}, min: {pred.min():.4f}, max: {pred.max():.4f}")

Prediction shape: (8, 1001), min: 0.1600, max: 0.8030


In [27]:
keras_loss = tf.keras.losses.BinaryCrossentropy()(y_dummy, pred).numpy()
draft_loss = binary_cross_entropy(tf.constant(y_dummy), tf.constant(pred)).numpy()
print(f"Keras BCE: {keras_loss:.6f}, draft BCE: {draft_loss:.6f}")
assert np.isclose(keras_loss, draft_loss, atol=1e-5)

Keras BCE: 0.713371, draft BCE: 0.713371


In [28]:
loss_before = net.model.evaluate(X_dummy, y_dummy, verbose=0)
net.model.fit(X_dummy, y_dummy, epochs=5, batch_size=BATCH, verbose=0)
loss_after = net.model.evaluate(X_dummy, y_dummy, verbose=0)
print(f"Loss before: {loss_before:.6f}, after 5 epochs on the dummy batch: {loss_after:.6f}")
assert loss_after < loss_before

Loss before: 0.801124, after 5 epochs on the dummy batch: 0.563158


In [29]:
PROC_DIR = '../data/processed'
REPORT_DIR = '../data/reports'
PRED_DIR = '../data/predictions'
MODEL_DIR = '../data/models'
VOCAB_SIZE = 1001
MAX_SEQ_LEN = 10
BATCH_SIZE = 256
MAX_EPOCHS = 40
PATIENCE = 5
RECENCY_DECAY = 0.7
SEED = 42
K_VALUES = [1, 3, 5, 9]
GRID = {
    'learning_rate': [1e-3, 3e-4],
    'dropout': [0.2, 0.3],
    'hidden_units': [128,256],
    'pos_weight' : [10.0, 20.0]
}
for d in [REPORT_DIR, PRED_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [30]:
X_padded = np.load(f'{PROC_DIR}/X_padded.npy').astype(np.float32)
Y = np.load(f'{PROC_DIR}/y_target.npy').astype(np.float32)
X_dense = np.load(f'{PROC_DIR}/X_rbm_dense.npy').astype(np.float32)
splits = np.load(f'{PROC_DIR}/splits.npz')
train_ex = splits['train']
val_ex = splits['validation']
test_ex = splits['test']

assert X_dense.shape[:2] == X_padded.shape[:2]
HIDDEN = X_dense.shape[2]
print(f"X_padded: {X_padded.shape}, X_dense: {X_dense.shape}, Y: {Y.shape}")
print(f"Examples - train: {len(train_ex)}, validation: {len(val_ex)}, test: {len(test_ex)}")

X_padded: (14074, 10, 1001), X_dense: (14074, 10, 500), Y: (14074, 1001)
Examples - train: 38981, validation: 14074, test: 14074


In [31]:
real_mask = (X_padded.sum(axis=2, keepdims=True) > 0).astype(np.float32)
H_hist = X_dense * real_mask
S_multi = np.concatenate([X_padded, Y[:, None, :]], axis=1)
print(f"Masked dense history: {H_hist.shape}, labels: {S_multi.shape}")

Masked dense history: (14074, 10, 500), labels: (14074, 11, 1001)


In [32]:
def dense_window(u, p):
    L = min(p, MAX_SEQ_LEN)
    x = np.zeros((MAX_SEQ_LEN, HIDDEN), dtype=np.float32)
    x[MAX_SEQ_LEN - L:] = H_hist[u, p - L:p]
    return x


def history_freq_window(u, p, decay=RECENCY_DECAY):
    L = min(p, MAX_SEQ_LEN)
    hist_baskets = S_multi[u, p - L:p, 1:]  # real items only, strictly prior to target p
    weights = (decay ** np.arange(L - 1, -1, -1, dtype=np.float32))[:, None]
    return np.log1p((weights * hist_baskets).sum(axis=0)).astype(np.float32)


def materialize(examples, drop_empty_targets=False):
    X = np.stack([dense_window(u, p) for u, p in examples]).astype(np.float32)
    H = np.stack([history_freq_window(u, p) for u, p in examples]).astype(np.float32)
    y = S_multi[examples[:, 0], examples[:, 1]].astype(np.float32)
    if drop_empty_targets:
        valid = y[:, 1:].sum(axis=1) > 0
        X, H, y = X[valid], H[valid], y[valid]
    return X, H, y


Xtr, Htr, ytr = materialize(train_ex, drop_empty_targets=True)
Xva, Hva, yva = materialize(val_ex, drop_empty_targets=False)
Xte, Hte, yte = materialize(test_ex, drop_empty_targets=False)
print(f"Train X: {Xtr.shape}, H: {Htr.shape}, y: {ytr.shape}")
print(f"Validation X: {Xva.shape}, H: {Hva.shape}, y: {yva.shape}")
print(f"Test X: {Xte.shape}, H: {Hte.shape}, y: {yte.shape}")

Train X: (32487, 10, 500), H: (32487, 1000), y: (32487, 1001)
Validation X: (14074, 10, 500), H: (14074, 1000), y: (14074, 1001)
Test X: (14074, 10, 500), H: (14074, 1000), y: (14074, 1001)


In [33]:
ITEM_DIM = VOCAB_SIZE - 1
ytr_items = ytr[:, 1:]
yva_items = yva[:, 1:]


def to_full(scores):
    return np.concatenate([np.zeros((len(scores), 1), np.float32), scores], axis=1)

In [34]:
def make_ds(X, H, y, shuffle):
    def gen():
        idx = np.random.permutation(len(X)) if shuffle else np.arange(len(X))
        for i in range(0, len(X), BATCH_SIZE):
            b = idx[i:i + BATCH_SIZE]
            yield (X[b], H[b]), y[b]
    spec = (
        (
            tf.TensorSpec((None, MAX_SEQ_LEN, HIDDEN), tf.float32),
            tf.TensorSpec((None, ITEM_DIM), tf.float32),
        ),
        tf.TensorSpec((None, ITEM_DIM), tf.float32),
    )
    return tf.data.Dataset.from_generator(gen, output_signature=spec).prefetch(tf.data.AUTOTUNE)


train_ds = make_ds(Xtr, Htr, ytr_items, shuffle=True)
val_ds = make_ds(Xva, Hva, yva_items, shuffle=False)

In [35]:
p = np.clip(ytr_items.mean(axis=0), 1e-6, 1 - 1e-6)
PRIOR_LOGIT = np.log(p / (1 - p)).astype(np.float32)
print(f"Output bias init: min={PRIOR_LOGIT.min():.2f}, max={PRIOR_LOGIT.max():.2f}, size={PRIOR_LOGIT.shape}")

Output bias init: min=-7.55, max=-2.71, size=(1000,)


In [36]:
class EpochLog(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        print(f"    epoch {epoch + 1:>2} | loss={logs.get('loss', float('nan')):.5f} | "
              f"val_loss={logs.get('val_loss', float('nan')):.5f}", flush=True)


class ValHitRatio(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best = -1.0
        self.best_weights = None
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        scores = to_full(self.model.predict([Xva, Hva], batch_size=BATCH_SIZE, verbose=0))
        res = evaluate_predictions(scores, yva, K_VALUES)
        mhr = float(np.mean([res[k]['HitRatio'] for k in K_VALUES]))
        print(f"    epoch {epoch + 1:>2} | val mean HR={mhr:.4f}", flush=True)
        if mhr > self.best:
            self.best = mhr
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= PATIENCE:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)


def train_config(cfg, device):
    with tf.device(device):
        net = LSTM(
            sequence_length=MAX_SEQ_LEN,
            input_dim=HIDDEN,
            output_dim=ITEM_DIM,
            hidden_units=cfg['hidden_units'],
            dropout=cfg['dropout'],
            learning_rate=cfg['learning_rate'],
            pos_weight=cfg['pos_weight'],
            use_history_residual=True,
        )
        callbacks = [
            EpochLog(),
            ValHitRatio(),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5
            ),
        ]
        hist = net.model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=MAX_EPOCHS,
            callbacks=callbacks,
            verbose=0,
        )
    return net, hist


def score_split(net, X, H):
    with tf.device(DEVICE):
        raw = net.model.predict([X, H], batch_size=BATCH_SIZE, verbose=0).astype(np.float32)
    return to_full(raw)

In [37]:
grid_results = []
nets = []
for lr, do, hu, pw in product(
    GRID['learning_rate'], GRID['dropout'], GRID['hidden_units'], GRID['pos_weight']
):
    cfg = {'learning_rate': lr, 'dropout': do, 'hidden_units': hu, 'pos_weight': pw}
    net, hist = train_config(cfg, DEVICE)
    val_scores = score_split(net, Xva, Hva)
    res = evaluate_predictions(val_scores, yva, K_VALUES)
    mean_hr = float(np.mean([res[k]['HitRatio'] for k in K_VALUES]))
    entry = {
        'config': cfg,
        'best_val_loss': float(min(hist.history['val_loss'])),
        'epochs_run': len(hist.history['loss']),
        'val_hit_ratio': {str(k): res[k]['HitRatio'] for k in K_VALUES},
        'val_mean_hit_ratio': mean_hr,
    }
    grid_results.append(entry)
    nets.append(net)
    print(f"lr={lr} dropout={do} hidden={hu} pos_w={pw} | val_loss={entry['best_val_loss']:.5f} "
          f"| val mean HR={mean_hr:.4f} | epochs={entry['epochs_run']}")

c:\Users\krpri\anaconda3\envs\virtualenv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
c:\Users\krpri\anaconda3\envs\virtualenv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


    epoch  1 | loss=0.18387 | val_loss=0.12988
    epoch  1 | val mean HR=0.1998
    epoch  2 | loss=0.13914 | val_loss=0.12973
    epoch  2 | val mean HR=0.1934
    epoch  3 | loss=0.13865 | val_loss=0.12964
    epoch  3 | val mean HR=0.1996
    epoch  4 | loss=0.13819 | val_loss=0.12964
    epoch  4 | val mean HR=0.1996
    epoch  5 | loss=0.13752 | val_loss=0.12876
    epoch  5 | val mean HR=0.2050
    epoch  6 | loss=0.13722 | val_loss=0.12838
    epoch  6 | val mean HR=0.2083
    epoch  7 | loss=0.13699 | val_loss=0.12802
    epoch  7 | val mean HR=0.2088
    epoch  8 | loss=0.13677 | val_loss=0.12823
    epoch  8 | val mean HR=0.2076
    epoch  9 | loss=0.13659 | val_loss=0.12777
    epoch  9 | val mean HR=0.2131
    epoch 10 | loss=0.13653 | val_loss=0.12811
    epoch 10 | val mean HR=0.2071
    epoch 11 | loss=0.13640 | val_loss=0.12783
    epoch 11 | val mean HR=0.2141
    epoch 12 | loss=0.13607 | val_loss=0.12752
    epoch 12 | val mean HR=0.2130
    epoch 13 | loss=0.13593 

In [38]:
best_idx = int(np.argmax([r['val_mean_hit_ratio'] for r in grid_results]))
best_cfg = grid_results[best_idx]['config']
best_net = nets[best_idx]
print(f"Selected on validation: {best_cfg}")
best_net.model.save(f'{MODEL_DIR}/lstm_best.keras')

Selected on validation: {'learning_rate': 0.001, 'dropout': 0.3, 'hidden_units': 256, 'pos_weight': 20.0}


In [39]:
test_scores = score_split(best_net, Xte, Hte)
test_res = evaluate_predictions(test_scores, yte, K_VALUES)

with open(f'{REPORT_DIR}/phase3_baseline_report.json') as f:
    base = json.load(f)

print(f"{'K':>4} | {'Model':<13} | {'HitRatio':>9} | {'Recall':>9}")
for k in K_VALUES:
    print(f"{k:>4} | {'LSTM':<13} | {test_res[k]['HitRatio']:>9.4f} | {test_res[k]['Recall']:>9.4f}")
    for label, key in [('Most Popular', 'most_popular'), ('FP-Growth', 'fpgrowth')]:
        r = base[key][str(k)]
        print(f"{k:>4} | {label:<13} | {r['HitRatio']:>9.4f} | {r['Recall']:>9.4f}")

for k in K_VALUES:
    for label, key in [('Most Popular', 'most_popular'), ('FP-Growth', 'fpgrowth')]:
        base_hr = base[key][str(k)]['HitRatio']
        lift = (test_res[k]['HitRatio'] - base_hr) / max(base_hr, 1e-12)
        print(f"K={k}: LSTM HitRatio lift over {label}: {lift:+.2%}")

   K | Model         |  HitRatio |    Recall
   1 | LSTM          |    0.1119 |    0.0417
   1 | Most Popular  |    0.1533 |    0.0707
   1 | FP-Growth     |    0.1533 |    0.0707
   3 | LSTM          |    0.2377 |    0.0975
   3 | Most Popular  |    0.1998 |    0.0927
   3 | FP-Growth     |    0.2034 |    0.0930
   5 | LSTM          |    0.3113 |    0.1348
   5 | Most Popular  |    0.2347 |    0.1044
   5 | FP-Growth     |    0.2374 |    0.1051
   9 | LSTM          |    0.3843 |    0.1724
   9 | Most Popular  |    0.2796 |    0.1219
   9 | FP-Growth     |    0.2829 |    0.1234
K=1: LSTM HitRatio lift over Most Popular: -27.02%
K=1: LSTM HitRatio lift over FP-Growth: -27.02%
K=3: LSTM HitRatio lift over Most Popular: +18.99%
K=3: LSTM HitRatio lift over FP-Growth: +16.87%
K=5: LSTM HitRatio lift over Most Popular: +32.65%
K=5: LSTM HitRatio lift over FP-Growth: +31.13%
K=9: LSTM HitRatio lift over Most Popular: +37.43%
K=9: LSTM HitRatio lift over FP-Growth: +35.83%


In [40]:
np.save(f'{PRED_DIR}/lstm_test_scores.npy', test_scores)
report = {
    'grid_results': grid_results,
    'selected_config': best_cfg,
    'test_lstm': {str(k): v for k, v in test_res.items()},
}
with open(f'{REPORT_DIR}/lstm_training_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(f"Saved test scores to {PRED_DIR} and report to {REPORT_DIR}")

Saved test scores to ../data/predictions and report to ../data/reports


In [41]:
s = test_scores[:, 1:]
item_mean = s.mean(axis=0)
total_var = s.var()
item_part = item_mean.var()
user_part = (s - item_mean).var()
print(f"Variance explained by item identity (popularity): {item_part / total_var:.2%}")
print(f"Variance explained by user (personalization):     {user_part / total_var:.2%}")

Variance explained by item identity (popularity): 66.47%
Variance explained by user (personalization):     33.53%
